<a href="https://colab.research.google.com/github/sui-ncsu/April26_Rutgers-RCSB/blob/main/004_Linear_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img align="right" src="https://github.com/codeBMB/April26_Rutgers-RCSB/raw/main/images/PyBMB_logo.png" width="150" height="150" />

# Project: April26_Rutgers-RCSB
## Notebook: 004-Linear regression

**Purpose:**
Linear regression is one of the most widely used tools in quantitative science. Whenever you prepare a standard curve — for a protein assay, an ELISA, or any experiment where you measure known standards to determine unknown values — you are applying linear regression. In this notebook you will build a reusable workflow using real Bradford protein assay data, from loading your data all the way to calculating protein concentrations in unknown samples. This notebook draws together skills from Notebooks 001–003.

In this lesson you'll learn to:

*   Load experimental data from a CSV file into a pandas DataFrame and inspect it for quality
*   Calculate means and standard deviations across replicate measurements
*   Build a scatter plot with error bars to visualize your standard curve
*   Perform linear regression using scipy.stats.linregress and interpret the slope, intercept, and R² value
*   Use the regression equation to calculate protein concentrations in unknown samples
*   Apply the same workflow to new datasets using both a dictionary and a DataFrame approach

**Input Data:**
* **Description:** Bradford assay standard curve data for BSA
  (bovine serum albumin) with three replicates at six concentrations
  (0.00 – 0.50 mg/mL), measured at 595 nm.
* **Source:** `bsa_std_curve.csv` — CodeBMB April 2025 workshop dataset
* **Retrieved On:** 2025-04-26
* **Access:** Downloaded automatically from the CodeBMB GitHub repository
  in Section 3a - https://raw.githubusercontent.com/codeBMB/April26_Rutgers-RCSB/main/data/bsa_std_curve.csv


**Libraries:**
* `pandas` — loading and organizing data in DataFrames
* `numpy` — numerical operations and generating best-fit line points
* `matplotlib` — creating and customizing plots
* `scipy.stats` — performing linear regression with `linregress`

**Status with Date:** April 2025 — Ready for workshop use

**License**

<img src="https://github.com/codeBMB/April26_Rutgers-RCSB/raw/main/images/by-nc-sa.png" width="100" alttext="[CC BY-NC-SA](https://creativecommons.org/licenses/by-nc-sa/4.0/)"/>
 This license enables reusers to distribute, remix, adapt, and build upon the material in any medium or format for noncommercial purposes only, and only so long as attribution is given to the creator. If you remix, adapt, or build upon the material, you must license the modified material under identical terms. CC BY-NC-SA includes the following elements:
 BY: credit must be given to the creator.
 NC: Only noncommercial uses of the work are permitted.
 SA: Adaptations must be shared under the same terms.

---
**Authorship: Zinedine Sehili, Paul Craig, Wally Novak**

**Acknowledgements: Charlie Weiss, SciCompforChemists**

**Contact Info:** codingBMB@gmail.com


# 0. Explanation of Colab and how to run (in notebooks for the first workshop)

To run the cell below, simply click the Run button located in that cell.

![run button image](https://github.com/wallynovak/biochemistry_seq_analysis/blob/main/images/run.png?raw=1)

NOTE: A cell is still running if you see a "stop" button with a moving circle around it. You can tell a cell has completed running as it will have a number in brackets on the left hand side (e.g. [1]) and a checkmark with the amount of time it took to run underneath it.

Please ensure every cell is done running before running the next cell.

# 1. Environment Setup & Libraries

## Libraries Used in This Notebook

This notebook uses four Python libraries. Run this cell first —
everything below depends on it.

| Library | Purpose |
|---|---|
| `pandas` | Loading the CSV file and organizing data into a table |
| `numpy` | Generating points for the best-fit line |
| `matplotlib` | Creating and customizing plots |
| `scipy.stats` | Performing linear regression with `linregress` |

<br>

In [ ]:
#Run this cell to import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# 2. Theory Block

## The Bradford Protein Assay

The **Bradford assay** is one of the most widely used methods in biochemistry
for measuring protein concentration. It relies on a dye —
**Coomassie Brilliant Blue G-250** — that binds to proteins and changes
color in the process. When the dye is free in solution it appears
reddish-brown, but when it binds to amino acid residues (particularly
arginine) in a protein, it shifts to a stable blue color.

We measure this color change using a **spectrophotometer** at a wavelength
of **595 nm**. The key principle is simple:

> **More protein → more dye binding → deeper blue color →
> higher absorbance at 595 nm**

Because we cannot measure protein concentration directly from the absorbance
reading, we need a **standard curve**: a set of solutions with *known*
protein concentrations (our BSA standards) that we use to build a reference.
Once we have the standard curve, we can use it to determine the concentration
of any *unknown* sample.

---

## Why Linear Regression?

The Bradford assay follows **Beer-Lambert Law**, which tells us that
absorbance is proportional to concentration:

$$A = \varepsilon \cdot l \cdot c$$

where:
- $A$ = absorbance (measured by the spectrophotometer)
- $\varepsilon$ = molar absorptivity (a constant for our dye-protein system)
- $l$ = path length (fixed by the cuvette, usually 1 cm)
- $c$ = concentration

Since $\varepsilon$ and $l$ are constants, this simplifies to a
**straight line** — just like the $y = mx + b$ you learned in algebra:

$$A_{595} = slope \times [Protein] + intercept$$

| Variable | Role | In our experiment |
|---|---|---|
| $y$ | Output (what we measure) | Absorbance at 595 nm |
| $m$ | Slope | Change in absorbance per mg/mL |
| $x$ | Input (what we control) | Protein concentration (mg/mL) |
| $b$ | Y-intercept | Absorbance when [Protein] = 0 |

**Linear regression** is the mathematical method that finds the *best
possible* values of slope and intercept to describe our standard curve. Python's `scipy.stats.linregress` function does this calculation for us —
we will use it in Section 5.4.

---

## What is R²?

The **coefficient of determination (R²)** tells us how well the straight
line fits our data. It ranges from 0 to 1:

| R² Value | What it means |
|---|---|
| R² = 1.00 | Perfect fit — every point lies exactly on the line |
| R² ≥ 0.98 | Excellent — expected for a good Bradford assay |
| R² ≥ 0.95 | Acceptable |
| R² < 0.90 | Poor — check for pipetting errors or a non-linear range |

> **In practice:** A well-prepared Bradford standard curve should have
> R² ≥ 0.98. Keep this benchmark in mind when you see your results
> in Section 5.4!

---

## Connecting to What You Already Know

In **Notebook 002**, you calculated velocities using the Michaelis-Menten
equation and stored them in a list. Here, we will do something similar:
- Our **known concentrations** (standards) play the role of `substrate_concs`
- Our **absorbance readings** play the role of `velocities`
- But instead of *calculating* the relationship from an equation, we will
  *fit* the equation to real experimental data

---

## Solving for Unknown Concentrations

Once we have slope and intercept from linear regression, we can
**rearrange** our equation to calculate concentration from any
measured absorbance:

Starting from:
$$A_{595} = slope \times [Protein] + intercept$$

Rearranging to solve for concentration:
$$[Protein] = \frac{A_{595} - intercept}{slope}$$

This rearranged equation is the practical payoff of this notebook —
we will use it in **Section 5.5** to find the concentrations of
unknown protein samples.

# 3. Data Acquisition

We will pull this data directly from GitHub; however, you could also upload a csv file to Colab directly.

## 3a. Downloading data from Github

In [ ]:
# Download the data file directly from GitHub
import urllib.request
url = 'https://raw.githubusercontent.com/codeBMB/April26_Rutgers-RCSB/main/data/bsa_std_curve.csv'
urllib.request.urlretrieve(url, 'bsa_std_curve.csv')
print("✅ Data file downloaded successfully!")

## 3b. Loading the Bradford Assay Data

Our data file, `bsa_std_curve.csv`, contains absorbance measurements
for a **BSA (bovine serum albumin) standard curve**. BSA is routinely
used as a protein standard because it is inexpensive, stable, and
well-characterized.

The file is organized as a spreadsheet with four columns:

| Column | Description |
|---|---|
| `Concentration_mg_per_mL` | Known BSA concentration (mg/mL) |
| `Rep1`, `Rep2`, `Rep3` | Absorbance readings at 595 nm for each replicate |

The code below loads this file into a **pandas DataFrame** called `df`.
You worked with DataFrames in **Notebook 003** — think of it as a
spreadsheet inside Python.

Run the cell and check that no error messages appear.

In [ ]:
# Define the filename by editing the variable fname
fname='bsa_std_curve.csv'
# Define the path to the CSV file in the /content folder
csv_filepath = '/content/' + fname

# Read the CSV file into a DataFrame named 'df'
df = pd.read_csv(csv_filepath)

#Ensure there is data in the dataframe
df.head()

# 4. Data Inspection & Cleaning

Before any analysis, we follow three steps:

1. **Look** at the data to confirm it loaded correctly
2. **Check** for missing values (NaN) that could cause calculation errors
3. **Clean** if necessary

`dropna()` removes any rows containing missing values. With our
carefully prepared dataset we don't expect any — but this step is
good practice for any real experiment where instruments sometimes
fail to record a value.

`clean_data_df.head(6)` displays all 6 rows of our standard curve.
Confirm that you can see concentrations from 0.00 to 0.50 mg/mL
and three replicate absorbance readings for each.


In [ ]:
# Perform cleaning and assign to a new variable name representing the stage
clean_data_df = df.dropna().copy()

clean_data_df.head(6)

# 5. Data Analysis

With our data loaded and cleaned, we will work through five steps:

1. Calculate the **mean absorbance** and **standard deviation**
   across the three replicates for each concentration
2. **Plot** the standard curve to visually inspect the data
3. Add **error bars** to show measurement variability
4. Perform **linear regression** and evaluate the fit with R²
5. Use the regression equation to calculate **unknown protein concentrations**

Run each cell in order and read the output before moving on.

## 5.1 Calculating the Average and Standard Deviation

Each concentration point was measured **three times** (Rep1, Rep2, Rep3).
Rather than work with three separate values, we collapse them into:
- **Average** — the central value we will plot and fit
- **Standard deviation** — the spread we will show as error bars

The key detail here is `axis=1`. In pandas:
- `axis=0` calculates *down* the columns (one result per column, this is the default)
- `axis=1` calculates *across* the rows (one result per row)

Since our three replicates are in three separate columns and we want
one average **per row** (per concentration), we use `axis=1`.

Run the cell below and confirm you see six rows with two new columns
added: `average` and `standard deviation`.

In [ ]:
# 5.1 Calculate the average and standard deviation for the replicate columns
# Create a list of the replicate columns
replicate_cols = ['Rep1', 'Rep2', 'Rep3']

# Calculate the average across the replicate columns for each row
clean_data_df['average'] = clean_data_df[replicate_cols].mean(axis=1)

# Calculate the standard deviation across the replicate columns for each row
clean_data_df['standard deviation'] = clean_data_df[replicate_cols].std(axis=1)

# Display the first few rows of the DataFrame with the new columns
clean_data_df.head(6)

## 📝 Exercise 5.1 — Exploring the Standard Curve Data

Look at the DataFrame output above and answer the following:

1. The absorbance at 0.00 mg/mL is slightly negative in one replicate.
   Why might this happen in a real experiment? Is it a problem?
2. In the code cell below, `print()` the average absorbance at the
   highest concentration point.
   *Hint: From Notebook 002, you know how to access a specific item
   by its index. With a DataFrame column you can write:
   `clean_data_df['average'][5]`*
3. Add a new column called `'CV_percent'` (coefficient of variation)
   using the formula below. CV% is a measure of how consistent your
   replicates are — lower is better.

$$CV\% = \frac{standard\ deviation}{average} \times 100$$

*Hint: Follow the same pattern used to calculate `'average'` and
`'standard deviation'` in the cell above.*

In [ ]:
# 1. Print the average absorbance at the highest concentration (index 5)


# 2. Create a new column 'CV_percent' in clean_data_df
#    CV_percent = (standard deviation / average) * 100


# Display the updated DataFrame
clean_data_df.head(6)

<details>
<summary>🔍 CLICK TO SEE ANSWER</summary>

```python
# 1. Print average at highest concentration
print(clean_data_df['average'][5])

# 2. Calculate CV%
clean_data_df['CV_percent'] = (
    clean_data_df['standard deviation'] / clean_data_df['average']
) * 100

clean_data_df.head(6)
```
Note: The CV% at 0.00 mg/mL will be very large or undefined
because we are dividing by a value near zero. This is expected —
CV% is not meaningful when the average is close to zero!
</details>

## 5.2 Plotting the Standard Curve

Before fitting any model, always **look at your data first**.
A scatter plot immediately shows us:
- Is the relationship between concentration and absorbance linear?
- Does the absorbance start near zero when concentration is zero?
- Are there any obvious outliers?

The plot below shows average absorbance on the y-axis and
concentration on the x-axis. You should see a clean linear
trend from bottom-left to top-right.

In [ ]:
# 5.2 Create the plot
plt.figure(figsize=(8, 6))
plt.plot(clean_data_df['Concentration_mg_per_mL'], clean_data_df['average'], 'o', label='Average Absorbance')

# Add labels and title
plt.xlabel('Concentration (mg/mL)')
plt.ylabel('Absorbance at 595 nm (Abs595)')
plt.title('Bradford Assay Standard Curve')
plt.grid(True)
plt.legend()
plt.show()

## 5.3 Adding Error Bars

Our assay used **three replicates** per concentration, giving us
information about the **consistency** of our measurements.

Error bars display the standard deviation at each point and tell us:
- How **reproducible** our pipetting was
- Whether variability changes across the concentration range
- If any point might be an **outlier**

A well-prepared Bradford assay will show small, tight error bars.
Look at how the error bar size changes from low to high concentrations.

In [ ]:
# 5.3 Add error bars to the plot

# Create the plot with error bars
plt.figure(figsize=(8, 6))
plt.errorbar(
    clean_data_df['Concentration_mg_per_mL'],
    clean_data_df['average'],
    yerr=clean_data_df['standard deviation'],
    fmt='o', # format as circles (no lines)
    capsize=4, # cap size for the error bars
    label='Average Absorbance with Std Dev'
)

# Add labels and title
plt.xlabel('Concentration (mg/mL)')
plt.ylabel('Absorbance at 595 nm (Abs595)')
plt.title('Bradford Assay Standard Curve with Error Bars')
plt.grid(True)
plt.legend()
plt.show()

## 5.4 Fitting the Line: Linear Regression

Now we fit a straight line to our data using `scipy.stats.linregress`.
This function takes our x values (concentration) and y values
(average absorbance) and returns five values:

| Variable | Meaning |
|---|---|
| `slope` | Change in absorbance per mg/mL of protein |
| `intercept` | Expected absorbance when concentration = 0 |
| `r_value` | Correlation coefficient (we square this to get R²) |
| `p_value` | Statistical significance of the relationship |
| `std_err` | Uncertainty in the slope estimate |

After running this cell you will see a red best-fit line added to
the plot, with the equation and R² shown in the legend.

> Compare your R² to the benchmarks in Section 2.
> How would you describe the quality of this standard curve?



## 5.4 Fitting the Line: Linear Regression

Now we fit a straight line to our data using `scipy.stats.linregress`.
But before we run the function, let's take a moment to understand
**what it does and what it returns** — and more importantly,
**how to figure that out for yourself** using the official documentation.

---

### What is `scipy.stats.linregress`?

`scipy` is a Python library built for scientific computing.
`stats` is a module within scipy that contains statistical functions.
`linregress` is a function within stats that performs
**ordinary least squares linear regression** — it finds the
straight line that minimizes the total squared distance between
the line and your data points.

We call it like this:

```python
from scipy import stats
result = stats.linregress(x, y)
```

where `x` is our concentration values and `y` is our
average absorbance values.

---

### How Do We Know What It Returns?

This is the most important skill in this section:
**how to read documentation to understand a function.**

The official scipy documentation for `linregress` can be found at:

[https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.linregress.html](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.linregress.html)


When you open that page you will see:

1. **Parameters** — what you pass IN to the function
2. **Returns** — what the function gives BACK to you
3. **Examples** — working code you can run immediately

> **Tip:** The Returns section is almost always the most
> important part for beginners. It tells you exactly what
> you get back and what each value means.

---

### What Does `linregress` Return?

According to the documentation, `linregress` returns a
`LinregressResult` object with five named values:

| Name | Type | What it means |
|---|---|---|
| `slope` | float | The slope of the best-fit line — change in y per unit x |
| `intercept` | float | The y-intercept — value of y when x = 0 |
| `rvalue` | float | The correlation coefficient r — NOT R² yet |
| `pvalue` | float | The p-value for the hypothesis test that slope = 0 |
| `stderr` | float | Standard error of the slope estimate |

> **Important:** The function returns `rvalue` — the
> correlation coefficient **r**, not R².
> To get R² you need to square it: `r_squared = rvalue**2`

---

### How Do We Unpack the Return Values?

Python allows us to assign all five return values at once
in a single line — this is called **unpacking**:

```python
slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
```

The names on the left (`slope`, `intercept`, etc.) are our
choice — we can call them anything we want.
The order on the left must match the order the function
returns them, which we know from the documentation.

Alternatively we can store the whole result and access
each part using dot notation:

```python
result = stats.linregress(x, y)
print(result.slope)
print(result.intercept)
print(result.rvalue)
```

Both approaches give exactly the same answer.
We will use the unpacking approach because it makes
the code easier to read.

---

### Using the Help Function in Colab

You can also access documentation directly inside Colab
without opening a browser. Run the cell below to see the
built-in help for `linregress`:

```python
# You can always access documentation directly in Colab
# using the help() function — try it for any function
# you want to understand better

from scipy import stats
help(stats.linregress)
```

Read through the output above. Notice:

- The **Parameters** section tells you what x and y should be
- The **Returns** section lists all five values and their meanings
- The **Notes** section explains the statistical assumptions
- The **Examples** section shows working code you can copy

> **Getting into the habit of reading documentation is one of
> the most important skills you can develop as a scientific coder.
> Every library you will ever use has documentation like this —
> scipy, pandas, matplotlib, BioPython, and beyond.**

Now let's run the regression and explore what we get back.

> Compare your R² to the benchmarks in Section 2.
> How would you describe the quality of this standard curve?


In [ ]:
# 5.4 Determine the best-fit line, R-squared, and equation

# Perform linear regression
concentration = clean_data_df['Concentration_mg_per_mL']
average_absorbance = clean_data_df['average']

slope, intercept, r_value, p_value, std_err = stats.linregress(concentration, average_absorbance)

# Calculate R-squared
r_squared = r_value**2

# Generate points for the best-fit line
line_x = np.linspace(concentration.min(), concentration.max(), 100)
line_y = slope * line_x + intercept

# Create the plot with error bars and the best-fit line
plt.figure(figsize=(8, 6))
plt.errorbar(
    concentration,
    average_absorbance,
    yerr=clean_data_df['standard deviation'],
    fmt='o', # format as circles (no lines)
    capsize=4, # cap size for the error bars
    label='Average Absorbance with Std Dev'
)

# Add the best-fit line
plt.plot(line_x, line_y, color='red', label=f'Best-fit line: y = {slope:.3f}x + {intercept:.3f}\nR^2 = {r_squared:.3f}')

# Add labels and title
plt.xlabel('Concentration (mg/mL)')
plt.ylabel('Absorbance at 595 nm (Abs595)')
plt.title('Bradford Assay Standard Curve with Best-fit Line')
plt.grid(True)
plt.legend()
plt.show()

## 📝 Exercise 5.4 — Interpreting Your Regression Results

`scipy.stats.linregress` gave us five variables.
Let's print them clearly and evaluate the fit quality.

In the code cell below:
1. Use `print()` and f-strings to display `slope`, `intercept`,
   `r_squared`, `p_value`, and `std_err` with clear labels.
   *Hint: from Notebook 002 — `print(f"Slope: {slope:.4f}")`*

2. Write an `if` statement that prints:
   - **"Excellent fit ✅"** if `r_squared >= 0.98`
   - **"Acceptable fit ⚠️"** if `r_squared >= 0.95`
   - **"Check your data ❌"** if `r_squared < 0.95`
   *Hint: you used if/elif/else in Notebook 002!*

3. In plain language, what does the **slope** value tell you
   about how the Bradford assay responds to protein?

In [ ]:
# 1. Print all regression statistics with labels
print(f"Slope:      ")   # complete these f-strings
print(f"Intercept:  ")
print(f"R²:         ")
print(f"P-value:    ")
print(f"Std Error:  ")
print()

# 2. Write an if/elif/else statement to evaluate the fit
#    if r_squared >= 0.98:
#        ...

<details>
<summary>🔍 CLICK TO SEE ANSWER</summary>

```python
# 1. Print regression statistics
print(f"Slope:      {slope:.4f} Abs595 per mg/mL")
print(f"Intercept:  {intercept:.4f}")
print(f"R²:         {r_squared:.4f}")
print(f"P-value:    {p_value:.2e}")
print(f"Std Error:  {std_err:.4f}")
print()

# 2. Evaluate the fit
if r_squared >= 0.98:
    print("Excellent fit ✅")
elif r_squared >= 0.95:
    print("Acceptable fit ⚠️")
else:
    print("Check your data ❌")

```

> **About the slope:** A slope of ~1.2 Abs595 per mg/mL means that
> adding 1 mg/mL of protein increases the absorbance by about 1.2 units.
> A steeper slope means the assay is more **sensitive** to changes
> in protein concentration.

</details>

## 5.5 Calculating Unknown Protein Concentrations

We now apply our regression equation to real unknown samples from
a **protein purification experiment**. Each sample was collected
at a different stage of the purification:

| Sample | Label | Description |
|---|---|---|
| CFE | Cell-Free Extract | All proteins from broken cells — total protein |
| FT | Flowthrough | Proteins that did NOT bind to the purification column |
| E1 | Elution 1 | First fraction where target protein was released |
| E2 | Elution 2 | Second elution fraction |

We will use **two approaches** to calculate concentrations:
1. A `for` loop over a **dictionary** (connecting back to Notebook 002)
2. A **pandas DataFrame** calculation (connecting back to Notebook 003)

Both give the same answer — compare the outputs!

We begin with the equation for our standard curve

$$
A_{595} = slope * ProtConc + intercept
$$

which we can rearrange to solve for protein concentration.

$$
ProtConc = \frac{A_{595} - intercept}{slope}
$$

Our final equation will then solve for concentration in mg/mL, based on the units for the slope (/mg/mL) and the unitless intercept.

Now let's put our unknown sample data in dictionary, write out the code for the equation and use some neat tricks from pandas to add the concentrations to our table. Unknown sample absorbances:

*   CFE = 0.492
*   Flowthrough = 0.427
*   Elution 1 = 0.284
*   Elution 2 = 0.199




In [ ]:
# --- Approach 1: Dictionary and for loop ---
# A dictionary stores key:value pairs — here sample name:absorbance
sample_dict = {
    "CFE" : 0.492,  # Cell-Free Extract
    "FT"  : 0.427,  # Flowthrough
    "E1"  : 0.284,  # Elution 1
    "E2"  : 0.199   # Elution 2
}

# Print a header row for our results table
print("Sample\tconcentration (mg/mL)")

# Loop through each sample name (key) and absorbance (value)
# Apply the rearranged regression equation: conc = (Abs - intercept) / slope
for key, value in sample_dict.items():
    protein_conc = (value - intercept) / slope
    print(key, "\t", round(protein_conc, 3))

Now let's calculate the same concentrations using a **pandas DataFrame**.
This approach produces a formatted table that is easier to read,
share, and export — connecting back to what you learned in **Notebook 003**.

Notice that both approaches give exactly the same concentrations.
Which do you find easier to read and work with?

In [ ]:
# --- Approach 2: pandas DataFrame ---

# Create two lists: sample names and their measured absorbances
samples       = ["CFE", "FT", "E1", "E2"]
sample_Abs595 = [0.492, 0.427, 0.284, 0.199]

# zip() pairs each sample name with its absorbance value
# pd.DataFrame() organizes these pairs into a table with named columns
sample_df = pd.DataFrame(
    list(zip(samples, sample_Abs595)),
    columns=['Sample', 'Sample_Abs']
)

# Apply the regression equation to every row at once
# pandas applies the calculation across the entire column automatically
sample_df['concentration_mg/mL'] = (
    (sample_df['Sample_Abs'] - intercept) / slope
).round(3)

# Display the finished table
sample_df

## 📝 Exercise 5.5 — Adding a New Unknown Sample

A colleague just handed you one more sample to analyze:

- **Wash fraction (W1):** Abs595 = 0.357

In the code cell below:
1. Add `"W1": 0.357` to the `sample_dict` dictionary
   and re-run the `for` loop to print all concentrations
   including W1.

2. Add W1 to `sample_df` by adding `"W1"` to the `samples`
   list and `0.357` to the `sample_Abs595` list, then
   re-run the DataFrame calculation.

3. Based on its absorbance and calculated concentration,
   where does W1 fall in the purification story?
   Does it make sense for a wash fraction to have this value?

In [ ]:
# 1. Add W1 to the dictionary and re-run the for loop
sample_dict = {
    "CFE" : 0.492,
    "FT"  : 0.427,
    "E1"  : 0.284,
    "E2"  : 0.199,
    # add W1 here
}

print("Sample\tConcentration (mg/mL)")
for key, value in sample_dict.items():
    protein_conc = (value - intercept) / slope
    print(key, "\t", round(protein_conc, 3))


Now let's do the same calculation using a **pandas DataFrame**.
This approach is more powerful when you have many samples or want
to save and export your results as a table — connecting back to
what you learned in **Notebook 003**.


In [ ]:
# 2. Add W1 to the DataFrame
samples      = ["CFE", "FT", "E1", "E2"]   # add "W1" here
sample_Abs595 = [0.492, 0.427, 0.284, 0.199]  # add 0.357 here

sample_df = pd.DataFrame(
    list(zip(samples, sample_Abs595)),
    columns=['Sample', 'Sample_Abs']
)
sample_df['concentration_mg/mL'] = (
    (sample_df['Sample_Abs'] - intercept) / slope
).round(3)

sample_df

<details>
<summary>🔍 CLICK TO SEE ANSWER</summary>

```python
# 1. Dictionary with W1 added
sample_dict = {
    "CFE" : 0.492,
    "FT"  : 0.427,
    "E1"  : 0.284,
    "E2"  : 0.199,
    "W1"  : 0.357
}

print("Sample\tConcentration (mg/mL)")
for key, value in sample_dict.items():
    protein_conc = (value - intercept) / slope
    print(key, "\t", round(protein_conc, 3))

# 2. DataFrame with W1 added
samples       = ["CFE", "FT", "E1", "E2", "W1"]
sample_Abs595 = [0.492, 0.427, 0.284, 0.199, 0.357]

sample_df = pd.DataFrame(
    list(zip(samples, sample_Abs595)),
    columns=['Sample', 'Sample_Abs']
)
sample_df['concentration_mg/mL'] = (
    (sample_df['Sample_Abs'] - intercept) / slope
).round(3)

sample_df
```

> **Thinking about the biology:** A wash fraction typically has
> a lower protein concentration than the CFE but may be higher
> than the elutions if non-specifically bound proteins were
> washed off the column. Where does your W1 value fall
> relative to the other fractions?

</details>


# 6. Results & Interpretation

Review your outputs from Section 5 and answer the following:

**About the standard curve:**
- What was your R² value? Does it meet the ≥ 0.99 benchmark?
- Look at the error bars — are they larger at high or low concentrations?
  What might cause this in a real experiment?

**About the unknown samples:**
The four samples came from a protein purification experiment.
Thinking about what each fraction represents:
- Which sample do you expect to have the **highest** concentration?
  Does your result match this expectation?
- The Elution fractions (E1, E2) should be **more pure** than the CFE,
  but are they more concentrated? Why might that be?

> **Key takeaway:** Linear regression is not just a mathematical tool —
> it connects your instrument readings to real biological quantities.
> The same workflow applies to any assay with a linear response:
> BCA protein assay, ELISA, enzyme activity curves, and more.


#7. Wrap-Up

Congratulations on completing Notebook 004!

In this notebook you:
- ✅ Loaded real experimental data from a CSV file using pandas
- ✅ Calculated means and standard deviations across replicates
- ✅ Built a publication-quality plot with error bars
- ✅ Performed linear regression with `scipy.stats.linregress`
- ✅ Evaluated fit quality using R²
- ✅ Used the regression equation to find protein concentrations
   in unknown samples

**The workflow you built here is a reusable template.**
The same steps apply to any experiment where you:
1. Measure a series of known standards
2. Fit a line to the standard curve
3. Use that line to calculate unknowns

**Connections across the workshop:**

| Notebook | Key concept | How it appeared in 004 |
|---|---|---|
| 001 | Colab environment | The platform we worked in |
| 002 | Variables, loops, if statements | `slope`, `intercept`; `for` loop in 5.5 |
| 003 | DataFrames, column calculations | `pd.read_csv()`, `df['column'].mean()` |
| 004 | Linear regression | Bringing it all together! |

# Optional Challenge Questions

These challenges extend the skills you practiced in this notebook.
They are independent of each other — pick the ones that interest you most!

| Challenge | Topic | Difficulty |
|---|---|---|
| C1 | Account for sample dilution | ⭐⭐ |
| C2 | Calculate and plot residuals | ⭐⭐⭐ |
| C3 | Apply the workflow to a new assay | ⭐⭐⭐⭐ |

## Challenge 1 — Correcting for Sample Dilution ⭐⭐

In a real experiment, samples are often **diluted** before measurement
to bring their absorbance into the range of the standard curve.
If a sample was diluted, you must multiply the calculated concentration
by the **dilution factor** to get the true concentration.

$$[Protein]_{true} = [Protein]_{measured} \times dilution\ factor$$

For example, a 1:10 dilution means the dilution factor = 10.

The samples in this experiment were diluted as follows:

| Sample | Dilution Factor |
|---|---|
| CFE | 10 |
| FT | 5 |
| E1 | 2 |
| E2 | 1 (undiluted) |

In the code cell below:
1. Create a dictionary called `dilution_factors` with the
   values above.
2. Add a new column to `sample_df` called `dilution_factor`
   using the dictionary values.
3. Add a third column called `true_concentration_mg/mL`
   that multiplies `concentration_mg/mL` by `dilution_factor`.
4. Display the updated DataFrame.

> **Think about it:** How does accounting for dilution change
> your interpretation of which fraction has the most protein?

In [ ]:
# Challenge 1 — Correcting for sample dilution
# We recreate sample_df here with the original 4 samples
# so this challenge works independently of Exercise 5.5

# Recreate the original 4-sample DataFrame
samples       = ["CFE", "FT", "E1", "E2"]
sample_Abs595 = [0.492, 0.427, 0.284, 0.199]
sample_df = pd.DataFrame(
    list(zip(samples, sample_Abs595)),
    columns=['Sample', 'Sample_Abs']
)
# Recalculate measured concentrations using regression results
sample_df['concentration_mg/mL'] = (
    (sample_df['Sample_Abs'] - intercept) / slope
).round(3)

# 1. Create a dictionary of dilution factors
#    key = sample name, value = dilution factor
dilution_factors = {
    # add your key:value pairs here
}

# 2. Add a dilution_factor column by matching sample names to the dictionary
#    Hint: sample_df['Sample'].map(dilution_factors)
sample_df['dilution_factor'] = None # replace None with your code

# 3. Calculate the true concentration
#    true concentration = measured concentration * dilution factor
sample_df['true_concentration_mg/mL'] = None # replace None with your code

# 4. Display the updated DataFrame
sample_df


<details>
<summary>🔍 CLICK TO SEE ANSWER</summary>

```python
# Challenge 2 answer

# 1. Dilution factors dictionary
dilution_factors = {
    "CFE" : 10,
    "FT"  : 5,
    "E1"  : 2,
    "E2"  : 1
}

# 2. Add dilution factor column
sample_df['dilution_factor'] = sample_df['Sample'].map(dilution_factors)

# 3. Calculate true concentration
sample_df['true_concentration_mg/mL'] = (
    sample_df['concentration_mg/mL'] * sample_df['dilution_factor']
)

# 4. Display
sample_df
```

> **Interpretation:** Once you account for dilutions, the CFE
> concentration is much higher than it first appeared. This makes
> biological sense — the cell-free extract contains all the protein
> from the original sample and was diluted the most before measurement.

</details>

## Challenge 2 — Evaluating the Fit with Residuals ⭐⭐⭐

R² gives us a single number to summarize fit quality, but it
can hide patterns. **Residuals** tell a richer story.

A residual is the difference between what the line **predicts**
and what was actually **measured**:

$$residual = measured\ value - predicted\ value$$

If the line fits well, residuals should be:
- **Small** — close to zero
- **Random** — no pattern (not all positive on one side, negative on the other)
- **Evenly spread** — similar size across the full concentration range

A pattern in residuals suggests the relationship may not be
perfectly linear, even if R² looks good.

In the code cell below:
1. Calculate the **predicted absorbance** for each standard
   concentration using your `slope` and `intercept`.
2. Calculate the **residuals** by subtracting predicted from measured.
3. Add both as new columns to `clean_data_df`.
4. Create a **residual plot**: concentration on the x-axis,
   residuals on the y-axis, with a horizontal dashed line at y=0.

> **Think about it:** Is there any pattern in the residuals?
> What would a curved pattern suggest about the Bradford assay
> at higher concentrations?

In [ ]:
# Challenge 2 — Residual plot

# 1. Calculate predicted absorbance for each standard
#    Use the regression equation: predicted = slope * concentration + intercept
clean_data_df['predicted'] = None  # replace None with your code

# 2. Calculate residuals
#    residual = measured (average) - predicted
clean_data_df['residual'] = None  # replace None with your code

# 3. Print the updated DataFrame to check your new columns
print(clean_data_df[['Concentration_mg_per_mL',
                      'average', 'predicted', 'residual']])

# 4. Create the residual plot
plt.figure(figsize=(8, 5))

# Plot residuals as scatter points
# Hint: plt.scatter(x, y, color='steelblue', s=80, zorder=5)


# Add a horizontal dashed line at y = 0
# Hint: plt.axhline(y=0, linestyle='--', color='red', alpha=0.7)


plt.xlabel('Concentration (mg/mL)')
plt.ylabel('Residual (measured − predicted)')
plt.title('Residual Plot — Bradford Standard Curve')
plt.grid(True)
plt.show()

<details>
<summary>🔍 CLICK TO SEE ANSWER</summary>

```python
# Challenge 3 answer

# 1. Predicted absorbance
clean_data_df['predicted'] = (
    slope * clean_data_df['Concentration_mg_per_mL'] + intercept
)

# 2. Residuals
clean_data_df['residual'] = (
    clean_data_df['average'] - clean_data_df['predicted']
)

# 3. Print to check
print(clean_data_df[['Concentration_mg_per_mL',                      'average', 'predicted', 'residual']])

# 4. Residual plot
plt.figure(figsize=(8, 5))

plt.scatter(
    clean_data_df['Concentration_mg_per_mL'],
    clean_data_df['residual'],
    color='steelblue',
    s=80,
    zorder=5,
    label='Residuals'
)

plt.axhline(y=0, linestyle='--', color='red', alpha=0.7, label='Zero line')

plt.xlabel('Concentration (mg/mL)')
plt.ylabel('Residual (measured − predicted)')
plt.title('Residual Plot — Bradford Standard Curve')
plt.grid(True)
plt.legend()
plt.show()
```

> With our clean synthetic data, residuals should be very small
> and show no obvious pattern — confirming a strong linear fit.
> In real Bradford data at higher concentrations (above ~1 mg/mL),
> you would often see residuals curve upward, indicating that the
> assay is becoming non-linear and that a linear model is no longer
> appropriate in that range.

</details>

## Challenge 3 — Apply the Workflow to BCA Assay Data ⭐⭐⭐⭐

The **BCA (bicinchoninic acid) assay** is another widely used protein
quantification method. Like the Bradford assay, it produces a linear
relationship between protein concentration and absorbance — but it
is measured at **562 nm** and uses a different concentration range
(typically 0–1000 μg/mL rather than 0–0.5 mg/mL).

The data below comes from a BCA standard curve experiment.

**Your task:** Apply the **complete workflow from this notebook**
to the BCA data. This means:

1. Create a DataFrame from the data below
2. Calculate average and standard deviation across replicates
3. Plot the standard curve with error bars
4. Perform linear regression and display slope, intercept, and R²
5. Use the regression equation to calculate concentrations
   for these unknown samples:
   - Sample X: Abs562 = 0.842
   - Sample Y: Abs562 = 1.456
   - Sample Z: Abs562 = 0.231

> **Before you start:** Look at the concentration units for
> the BCA data. How are they different from the Bradford data?
> How will this affect your axis labels and your interpretation
> of the slope?

In [ ]:
# Challenge 3 — BCA Assay Workflow

# BCA standard curve data
# Note: concentrations are in micrograms per mL (μg/mL)
bca_data = {
    'Concentration_ug_per_mL': [0, 25, 125, 250, 500, 750, 1000],
    'Rep1': [0.131, 0.193, 0.379, 0.645, 1.165, 1.687, 2.199],
    'Rep2': [0.128, 0.197, 0.376, 0.650, 1.170, 1.693, 2.196],
    'Rep3': [0.130, 0.191, 0.383, 0.642, 1.163, 1.680, 2.203]
}

# Step 1: Create a DataFrame from bca_data


# Step 2: Calculate average and standard deviation across replicates


# Step 3: Plot the standard curve with error bars


# Step 4: Perform linear regression and print slope, intercept, R²


# Step 5: Calculate concentrations for the unknown samples
bca_unknowns = {
    "Sample X": 0.842,
    "Sample Y": 1.456,
    "Sample Z": 0.231
}
# print results using a for loop

<details>
<summary>🔍 CLICK TO SEE ANSWER</summary>

```python
# Challenge 4 answer

# Step 1: Create DataFrame
bca_data = {
    'Concentration_ug_per_mL': [0, 25, 125, 250, 500, 750, 1000],
    'Rep1': [0.131, 0.193, 0.379, 0.645, 1.165, 1.687, 2.199],
    'Rep2': [0.128, 0.197, 0.376, 0.650, 1.170, 1.693, 2.196],
    'Rep3': [0.130, 0.191, 0.383, 0.642, 1.163, 1.680, 2.203]
}
bca_df = pd.DataFrame(bca_data)

# Step 2: Average and standard deviation
bca_cols = ['Rep1', 'Rep2', 'Rep3']
bca_df['average'] = bca_df[bca_cols].mean(axis=1)
bca_df['std_dev']  = bca_df[bca_cols].std(axis=1)
print(bca_df[['Concentration_ug_per_mL', 'average', 'std_dev']])

# Step 3: Plot with error bars
plt.figure(figsize=(8, 6))
plt.errorbar(
    bca_df['Concentration_ug_per_mL'],
    bca_df['average'],
    yerr=bca_df['std_dev'],
    fmt='o',
    capsize=4,
    label='BCA Standards'
)
plt.xlabel('Concentration (μg/mL)')
plt.ylabel('Absorbance at 562 nm')
plt.title('BCA Assay Standard Curve')
plt.grid(True)
plt.legend()
plt.show()

# Step 4: Linear regression
bca_conc = bca_df['Concentration_ug_per_mL']
bca_abs  = bca_df['average']

bca_slope, bca_intercept, bca_r, bca_p, bca_se = stats.linregress(
    bca_conc, bca_abs
)
bca_r2 = bca_r**2

print(f"\nSlope:     {bca_slope:.6f} Abs562 per μg/mL")
print(f"Intercept: {bca_intercept:.4f}")
print(f"R²:        {bca_r2:.4f}")

# Plot with best-fit line
bca_line_x = np.linspace(bca_conc.min(), bca_conc.max(), 100)
bca_line_y = bca_slope * bca_line_x + bca_intercept

plt.figure(figsize=(8, 6))
plt.errorbar(bca_conc, bca_abs, yerr=bca_df['std_dev'],
             fmt='o', capsize=4, label='BCA Standards')
plt.plot(bca_line_x, bca_line_y, color='red',
         label=f'Best-fit: y = {bca_slope:.5f}x + {bca_intercept:.3f}\nR² = {bca_r2:.4f}')
plt.xlabel('Concentration (μg/mL)')
plt.ylabel('Absorbance at 562 nm')
plt.title('BCA Assay Standard Curve with Best-fit Line')
plt.grid(True)
plt.legend()
plt.show()

# Step 5: Unknown concentrations
bca_unknowns = {
    "Sample X": 0.842,
    "Sample Y": 1.456,
    "Sample Z": 0.231
}

print("\nSample\tConcentration (μg/mL)")
for name, abs_val in bca_unknowns.items():
    conc = (abs_val - bca_intercept) / bca_slope
    print(f"{name}\t{conc:.1f}")
```

> **Key differences from Bradford:**
> - Concentration units are μg/mL instead of mg/mL
>   (1 mg/mL = 1000 μg/mL)
> - The slope is much smaller (~0.002 vs ~1.2) because
>   a large change in concentration produces a smaller
>   change in absorbance per unit
> - The linear range extends much higher (up to 1000+ μg/mL)
>
> Despite these differences, the **workflow is identical** —
> this is the power of building a reusable template!

</details>

## Notebook Sign-Off Checklist

* [ ] **Purpose is clear** from the header without opening code cells.
* [ ] **Every significant decision** has a markdown explanation.
* [ ] **Data source and extraction date** are recorded in the header.
* [ ] **Assumptions and exclusions** are explicitly stated.
* [ ] **Variable names** indicate both the content and workflow stage.
* [ ] **Kernel restarted** and all cells run top-to-bottom without error.